Optimization Exercises

Run vanilla gradient descent on the Rosenbrock function with learning rates [0.0001, 0.0005, 0.001, 0.005, 0.01]. Print the final loss after 5000 steps for each. Find the largest learning rate that still converges.

In [5]:
def rosenbrock(params):
    x, y = params
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2

##Definition of partial derivatives
def rosenbrock_gradient(params):
    x, y = params
    df_dx = -2 * (1 - x) + 200 * (y - x ** 2) * (-2 * x)
    df_dy = 200 * (y - x ** 2)
    return [df_dx, df_dy]

##definition of the gradient descent
class GradientDescent:
    def __init__(self, lr=0.001): ##initialization of class and def of learning rate
        self.lr = lr
 ##returns new values of x and y taking into account
      ## their grads and lr defined above
    def step(self, params, grads):
        return [p - self.lr * g for p, g in zip(params, grads)]

class SGDMomentum:
  ##Momentum class for SDG is defined and initialized
  ## lr, momentum and velocity are initialized
    def __init__(self, lr=0.001, momentum=0.9):
        self.lr = lr
        self.momentum = momentum
        self.velocity = None

#steps are defined considering momentum theory
#velocity is initially set as 0, only at the very beginning
## velocity then calculated for all grads
    def step(self, params, grads):
        if self.velocity is None:
            self.velocity = [0.0] * len(params) ##sets velocity as 0\

            ##takes v=0 and starts calculating velocity for all grads
        self.velocity = [
            self.momentum * v + g
            for v, g in zip(self.velocity, grads)
        ]

        ##returns parameters - lr * velocity for all
        ##params and velocities zipped.
        return [p - self.lr * v for p, v in zip(params, self.velocity)]

      ##adam class initialization
class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):

      ##initialization of m,v and t. lr, beta1,2 and epsilon values are saved.
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0


    def step(self, params, grads):
        if self.m is None:

          ##m and v are set to 0
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)

        ##count of t increasing by one
        self.t += 1

        self.m = [
            self.beta1 * m + (1 - self.beta1) * g
            for m, g in zip(self.m, grads)
        ] ## m = beta1*m + (1-beta1)*gradient
        self.v = [
            self.beta2 * v + (1 - self.beta2) * g ** 2
            for v, g in zip(self.v, grads)
        ] ## v = beta2*v + (1-beta2)* (gradient^2)

        ##definition of mhat and vhat.
        m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m] ## m/1-(b1^t)
        v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v] ## v/1-(b2^t)

        return [
            p - self.lr * mh / (vh ** 0.5 + self.epsilon)
            for p, mh, vh in zip(params, m_hat, v_hat)
        ] ##returns new position after applying adam, integrating vh, mh and epsilon


#defines an optimizer to run, 5000 steps.
def optimize(optimizer, func, grad_func, start, steps=5000):
  ##optimizer param is where we'll call either class previously defined
  ##func: loss function passed to the function
  ##grad_func: func that calculates gradients (eg: grad rosenrbrock)
  ##start: initial positions of x and y
    params = list(start) ##uses list to make a copy of start and uses this as initial x and y
    history = [params[:]] ## as code runs creates a copy with all entries of params
    for _ in range(steps):
        try:
          ## uses params to calculate grads
            grads = grad_func(params)
            ##uses the class through optimizer and calculates step using params and grads
            params = optimizer.step(params, grads)
            history.append(params[:]) ##appends position to history list
        except OverflowError:
            return None ## If it overflows, return None
    return history ##returns history

start = [-1.0, 1.0] ##defines start positions


##testing all classes/optimization methods
gd_history1 = optimize(GradientDescent(lr=0.0001), rosenbrock, rosenbrock_gradient, start)
gd_history2 = optimize(GradientDescent(lr=0.0005), rosenbrock, rosenbrock_gradient, start)
gd_history3 = optimize(GradientDescent(lr=0.001), rosenbrock, rosenbrock_gradient, start)
gd_history4 = optimize(GradientDescent(lr=0.005), rosenbrock, rosenbrock_gradient, start)
gd_history5 = optimize(GradientDescent(lr=0.1), rosenbrock, rosenbrock_gradient, start)

print("")
for name, history in [("GD_0.0001", gd_history1), ("GD_0.0005", gd_history2), ("GD_0.001", gd_history3), ("GD_0.005", gd_history4), ("GD_0.1", gd_history5)]:
    if history is None:
        print(f"{name:9s} -> Diverged / Overflow Error due to large learning rate.")
    else:
        final = history[-1]
        loss = rosenbrock(final) ##uses the last position on the rosenbrock function for the loss
        print(f"{name:9s} -> x={final[0]:.6f}, y={final[1]:.6f}, loss={loss:.8f}")


GD_0.0001 -> x=-0.446368, y=0.206459, loss=2.09718513
GD_0.0005 -> x=0.798131, y=0.636104, loss=0.04083385
GD_0.001  -> x=0.939826, y=0.883023, loss=0.00362714
GD_0.005  -> Diverged / Overflow Error due to large learning rate.
GD_0.1    -> Diverged / Overflow Error due to large learning rate.


Momentum comparison. Run SGD with momentum values [0.0, 0.5, 0.9] on the Rosenbrock function. Track the loss at every step. Which momentum value converges fastest? Which overshoots?

In [28]:
sgd_history1 = optimize(SGDMomentum(lr=0.0001, momentum=0.0), rosenbrock, rosenbrock_gradient, start)
sgd_history2 = optimize(SGDMomentum(lr=0.0001, momentum=0.5), rosenbrock, rosenbrock_gradient, start)
sgd_history3 = optimize(SGDMomentum(lr=0.0001, momentum=0.9), rosenbrock, rosenbrock_gradient, start)

for name, history in [
    ("momentum=0.0", sgd_history1),
    ("momentum=0.5", sgd_history2),
    ("momentum=0.9", sgd_history3)
]:
    print(f"\n{name}")
    for step, point in enumerate(history):
        if step % 1000 == 0:
            loss = rosenbrock(point)
            print(f"Step {step}: loss = {loss:.8f}")


momentum=0.0
Step 0: loss = 4.00000000
Step 1000: loss = 3.66469358
Step 2000: loss = 3.31657738
Step 3000: loss = 2.94420063
Step 4000: loss = 2.54058146
Step 5000: loss = 2.09718513

momentum=0.5
Step 0: loss = 4.00000000
Step 1000: loss = 3.31734464
Step 2000: loss = 2.54156820
Step 3000: loss = 1.61295736
Step 4000: loss = 0.76148196
Step 5000: loss = 0.40619883

momentum=0.9
Step 0: loss = 4.00000000
Step 1000: loss = 0.41065399
Step 2000: loss = 0.07353412
Step 3000: loss = 0.02350980
Step 4000: loss = 0.00881502
Step 5000: loss = 0.00355685


 Define the function
f(x,y) = x^2 - y^2 (a saddle point at the origin). Start at (0.01, 0.01). Compare how vanilla GD, SGD with momentum, and Adam behave. Which escapes the saddle point?

In [31]:
def funcxy(params):
  x, y = params
  return (x) ** 2 - (y) ** 2

def funcxy_gradient(params):
    x, y = params
    df_dx = 2*x
    df_dy = -2*y
    return [df_dx, df_dy]

start = [0.01, 0.01]

gd_historyxy = optimize(GradientDescent(lr=0.0005), funcxy, funcxy_gradient, start)
sgd_historyxy = optimize(SGDMomentum(lr=0.0001, momentum=0.9), funcxy, funcxy_gradient, start)
adam_historyxy = optimize(Adam(lr=0.01), funcxy, funcxy_gradient, start)

for name, history in [("GD", gd_historyxy), ("SGD+M", sgd_historyxy), ("Adam", adam_historyxy)]:
    final = history[-1]
    loss = funcxy(final)
    print(f"{name:6s} -> x={final[0]:.6f}, y={final[1]:.6f}, loss={loss:.8f}")




GD     -> x=0.000067, y=1.480428, loss=-2.19166813
SGD+M  -> x=0.000000, y=180.287597, loss=-32503.61758653
Adam   -> x=0.000273, y=68.315525, loss=-4667.01093275


x approached zero while y increased noticeably compared to its starting value. All three methods escaped the saddle point